# Best-response attackers on Blotto
For every frozen defender, a fresh recurrent-PPO attacker is trained from scratch against it under one shared budget, following the approximate-exploitability protocol of Timbers et al. (2022). These attackers are the fourth member of the evaluation suite in `eval.ipynb`.

- **Attacker interface**, identical for every defender: a GRU over [true state, previous allocation / B], conditioned on the full $\theta$, with its logits masked to the admissible allocations.
- **Defenders**: `Critic P1` (the Oracle, the critic's equilibrium row with privileged access to the state and context), `Trained P1` (our method), `RPPO`, `DR_RPPO`, `VARIBAD`, `PEARL`, `MMD`, and `Random P1`. Each defender samples its action from its own distribution, as at deployment.
- **Output**: `compared/BR_attacker_<defender>.pth`, holding the weights, the configuration, and the defender's reward per step for each update.

In [ ]:
%run Initialize.ipynb

## Settings

In [ ]:
BR_TOTAL_EPISODES = 5000     # same budget for every defender
EPISODES_PER_UPD  = 10
GAMMA             = 0.95
GAE_LAMBDA        = 0.95
PPO_EPOCHS        = 4
CLIP_EPS          = 0.2
ENT_COEF          = 0.01
VF_COEF           = 0.5
LR                = 3e-4
CKPT_EVERY        = 200
SEED_BASE         = 777
SAVE_DIR          = "compared"
os.makedirs(SAVE_DIR, exist_ok=True)

## Frozen defenders (the wrappers are defined in `Initialize.ipynb`)

In [ ]:
Context_Model = Load(TSTransformer().to(device), "encoder.pth")
Z_S_ENC       = Load(StateGRU().to(device), "z_s_enc.pth")
critic        = Load(Critic().to(device), "critic.pth")
Player1       = Load(Player1Actor().to(device), "player1.pth")
BASELINES     = Load_Baselines(SAVE_DIR)

print("Loaded baselines:", list(BASELINES.keys()))
DEFENDERS = ['Critic P1', 'Trained P1', 'RPPO', 'DR_RPPO', 'VARIBAD', 'PEARL', 'MMD', 'Random P1']

## Rollouts and PPO (`BRAttacker` is defined in `Initialize.ipynb`)

In [ ]:
def gae(rews, vals):
    T = len(rews)
    adv = np.zeros(T, dtype=np.float32); last = 0.0
    for t in reversed(range(T)):
        nv = vals[t + 1] if t + 1 < T else 0.0
        delta = rews[t] + GAMMA * nv - vals[t]
        last = delta + GAMMA * GAE_LAMBDA * last
        adv[t] = last
    return adv, adv + np.asarray(vals, dtype=np.float32)


def rollout_br(attacker, defw):
    Theta, Weights, Reachable, x, legal = sample_config()
    Type_Theta = np.concatenate([Weights, Reachable])
    defw.reset(Theta, Type_Theta, legal)
    legal_mask = torch.zeros(n_u, dtype=torch.bool, device=device)
    legal_mask[legal] = True
    theta_t = torch.tensor(Theta, dtype=torch.float32, device=device).view(1, 1, Len_Theta)
    h = None
    u_prev = np.zeros(N)
    o_l, u_l, lp_l, v_l, ar_l, dr_l = [], [], [], [], [], []
    for t in range(max_steps):
        a_dist = defw.act_dist(x)
        a = int(np.random.choice(n_a, p=a_dist))            # the defender samples from its own policy
        obs = np.concatenate([x, u_prev]).astype(np.float32)
        with torch.no_grad():
            logits, val, h = attacker(torch.tensor(obs, device=device).view(1, 1, 2 * N),
                                      theta_t, legal_mask.view(1, 1, n_u), h)
        dist = Categorical(logits=logits.view(-1))
        u = int(dist.sample())
        lp = float(dist.log_prob(torch.tensor(u, device=device)))
        a_vec, u_vec = actions[a], actions[u]
        nx = next_state(x, a_vec, u_vec, Theta)
        r = reward(nx, Theta)
        y = Observation(nx)
        defw.observe(x, a_vec, u_vec, nx, y)
        o_l.append(obs); u_l.append(u); lp_l.append(lp); v_l.append(val.item())
        ar_l.append(-r); dr_l.append(r)                      # the attacker maximizes -r
        x = nx; u_prev = u_vec / B
    adv, ret = gae(np.array(ar_l, dtype=np.float32), np.array(v_l, dtype=np.float32))
    return {'obs': np.stack(o_l), 'acts': np.array(u_l), 'logp': np.array(lp_l, dtype=np.float32),
            'adv': adv, 'ret': ret, 'theta': Theta.astype(np.float32), 'legal_mask': legal_mask,
            'def_rews': np.array(dr_l, dtype=np.float32)}


def ppo_update_br(att, opt, eps):
    S = lambda k, dt=torch.float32: torch.tensor(np.stack([e[k] for e in eps]), dtype=dt, device=device)
    obs, acts, oldlp, adv, ret = S('obs'), S('acts', torch.long), S('logp'), S('adv'), S('ret')
    theta = S('theta').unsqueeze(1).expand(-1, max_steps, -1)
    mask = torch.stack([e['legal_mask'] for e in eps]).unsqueeze(1).expand(-1, max_steps, -1)
    adv = (adv - adv.mean()) / (adv.std() + 1e-8)
    for _ in range(PPO_EPOCHS):
        logits, values, _ = att(obs, theta, mask)
        dist = Categorical(logits=logits)
        lp = dist.log_prob(acts)
        ratio = (lp - oldlp).exp()
        pg = -torch.min(ratio * adv, torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * adv).mean()
        vl = F.mse_loss(values, ret)
        loss = pg + VF_COEF * vl - ENT_COEF * dist.entropy().mean()
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(att.parameters(), 1.0)
        opt.step()


def train_br(def_name, seed):
    np.random.seed(seed); torch.manual_seed(seed); random.seed(seed)
    defw = get_defender(def_name)
    att = BRAttacker().to(device)
    opt = optim.AdamW(att.parameters(), lr=LR)
    n_upd = BR_TOTAL_EPISODES // EPISODES_PER_UPD
    def_hist = []
    fname = 'BR_attacker_' + def_name.replace(' ', '_') + '.pth'
    for upd in tqdm(range(n_upd), desc='BR vs ' + def_name):
        eps = [rollout_br(att, defw) for _ in range(EPISODES_PER_UPD)]
        ppo_update_br(att, opt, eps)
        def_hist.append(float(np.mean([e['def_rews'].mean() for e in eps])))
        if (upd + 1) % CKPT_EVERY == 0 or upd == n_upd - 1:
            torch.save({'attacker': att.state_dict(), 'defender': def_name,
                        'config': {'RNN_H': RNN_H, 'seed': seed, 'budget_episodes': BR_TOTAL_EPISODES},
                        'defender_reward_per_step': def_hist},
                       os.path.join(SAVE_DIR, fname))
    return att, def_hist

## Train one attacker per defender

In [ ]:
BR_ATTACKERS, BR_CURVES = {}, {}
for k, name in enumerate(DEFENDERS):
    BR_ATTACKERS[name], BR_CURVES[name] = train_br(name, seed=SEED_BASE + k)

## Convergence check: each defender's reward should plateau, otherwise increase `BR_TOTAL_EPISODES`

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
for name, c in BR_CURVES.items():
    ax.plot(np.arange(1, len(c) + 1), c, label=name)
ax.set_xlabel('PPO update'); ax.set_ylabel("defender's reward per step")
ax.grid(alpha=0.3); ax.legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.savefig('br_curves.jpg', dpi=300, bbox_inches='tight'); plt.show()